# 14. 교통안전 인프라 사각지대 도출

## 목적
229개 시군구 AI 예측 결과 중, 횡단보도·신호등·어린이보호구역·CCTV가 공통 확보된 **49개 상세지역**을 대상으로 `위험은 높은데 인프라는 부족한 지역`을 찾는다.

기존 EDA의 `infrastructure_score`는 네 가지 시설지표의 지역 내 백분위 점수를 평균한 값이다.


In [1]:
import pandas as pd
from pathlib import Path
BASE = Path("..")
risk = pd.read_csv(BASE / "results" / "predicted_high_risk_2026_229.csv")
eda = pd.read_csv(BASE / "data_processed" / "eda_main_49.csv")
print("AI 예측 지역:", len(risk))
print("상세 인프라 지역:", len(eda))


AI 예측 지역: 229
상세 인프라 지역: 49


### 1. AI 위험도와 인프라 공급수준 결합
- 인프라 부족도 = `1 - infrastructure_score`
- 개선 우선순위 = `(AI 위험도 + 인프라 부족도) / 2`

동일가중치는 설명 가능한 정책 우선순위용 휴리스틱이다. 인과효과나 최적예산배분 결과는 아니다.


In [2]:
blind = eda.merge(
    risk[["sido_sgg_nm", "predicted_high_risk_probability_2026"]],
    on="sido_sgg_nm", how="inner"
).copy()
blind["infrastructure_deficiency_score"] = 1 - blind["infrastructure_score"]
blind["improvement_priority_score"] = (
    blind["predicted_high_risk_probability_2026"] + blind["infrastructure_deficiency_score"]
) / 2
blind = blind.sort_values("improvement_priority_score", ascending=False).reset_index(drop=True)
blind.insert(0, "improvement_priority_rank", range(1, len(blind) + 1))

cols = [
    "improvement_priority_rank", "sido_sgg_nm", "predicted_high_risk_probability_2026",
    "infrastructure_score", "infrastructure_deficiency_score", "improvement_priority_score",
    "crosswalk_per_1000_child", "traffic_light_per_1000_child",
    "schoolzone_per_1000_child", "cctv_install_rate"
]
blind[cols].to_csv(BASE / "results" / "infrastructure_blind_spots_49.csv", index=False, encoding="utf-8-sig")
display(blind[cols].head(15))


,improvement_priority_rank,sido_sgg_nm,predicted_high_risk_probability_2026,infrastructure_score,infrastructure_deficiency_score,improvement_priority_score,crosswalk_per_1000_child,traffic_light_per_1000_child,schoolzone_per_1000_child,cctv_install_rate
0,1,강원특별자치도 정선군,0.923001,0.428571,0.571429,0.747215,27.388535,10.191083,11.464968,0.888889
1,2,강원특별자치도 태백시,0.983254,0.489796,0.510204,0.746729,1.759789,23.757149,5.719314,1.000000
2,3,충청남도 태안군,0.975390,0.484694,0.515306,0.745348,9.887487,1.363791,8.182748,1.000000
3,4,경상남도 통영시,0.894326,0.438776,0.561224,0.727775,39.292731,16.699411,5.034381,0.878049
4,5,충청북도 제천시,0.976913,0.525510,0.474490,0.725701,10.198364,69.147148,4.931077,0.954545
5,6,경상북도 영덕군,0.985369,0.540816,0.459184,0.722276,7.513661,24.590164,6.830601,1.000000
6,7,전라남도 담양군,0.981943,0.571429,0.428571,0.705257,8.261049,21.065675,11.978521,1.000000
7,8,경상북도 상주시,0.845352,0.469388,0.530612,0.687982,2.512563,27.831465,11.982992,0.887097
8,9,전라남도 여수시,0.764300,0.469388,0.530612,0.647456,20.123549,20.123549,4.913890,0.961905
9,10,전라남도 해남군,0.805005,0.581633,0.418367,0.611686,30.834512,18.104668,7.072136,1.000000


### 해석
상위 지역은 **AI가 상대적으로 고위험으로 판단하면서 현재 확보된 교통안전 인프라 수준은 상대적으로 낮은 지역**이다.

따라서 해당 지역을 시설 설치 대상으로 자동 확정하는 것이 아니라, **현장조사와 예산 검토를 우선할 후보지역**으로 제안한다.
